In [19]:
from FlagEmbedding import FlagReranker
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,

)
from sentence_transformers.losses import MultipleNegativesRankingLoss, TripletLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import TripletEvaluator, RerankingEvaluator
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)


# reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
import time

start = time.time()
# You can map the scores into 0-1 by set "normalize=True", which will apply sigmoid function to the score
# print(test_dataset['corpus']["1"])
scores = reranker.compute_score([[' đã được tiền huấn luyện trên bộ dữ liệu về y tế ', 'Retriever đau bụng bao gồm mô hình PhoBERT-base-v2 đã được tiền huấn luyện trên bộ dữ liệu về y tế - sức khỏe, mô hình bkai-foundation-models/vietnamese-bi-encoder và mô hình multilingual-e5-base đã được tinh chỉnh trên tác vụ truy xuất thông tin về y tế - sức khỏ']], normalize=True)
end = time.time()
print(scores) # [0.00027803096387751553, 0.9948403768236574]
print(end - start)
import torch
import json
embedder = SentenceTransformer('thang1943/multilingual-e5-large-v2')

with open('/mnt/data1tb/thangcn/datnv2/data/alobs_test_dataset.json', encoding='utf8') as f:
    test_dataset = json.load(f)
corpus = []
corpus_ids = []
for id in test_dataset['corpus'].keys():
    corpus.append(test_dataset['corpus'][id])
    corpus_ids.append(id)

corpus_embeddings = embedder.encode(corpus, convert_to_tensor=True)
queries = []
query_ids = []
for id in test_dataset['queries'].keys():
    queries.append(test_dataset['queries'][id])
    query_ids.append(id)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[0.9830200077626816]
0.4209132194519043


In [21]:
from langchain_core.documents import Document  
from langchain.vectorstores import FAISS

In [33]:
documents = [Document(page_content=text, metadata={"doc_id": id}) for text, id in zip(corpus,corpus_ids)]

In [24]:
from langchain_community.vectorstores import FAISS
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore

In [27]:
embeddings = HuggingFaceEmbeddings(
    model_name='thang1943/multilingual-e5-large-v2',
    model_kwargs={'device': 'cuda'}
)

/tmp/ipykernel_236963/447660529.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [35]:
index = faiss.IndexFlatL2(len(embeddings.embed_query('thang')))
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    index_to_docstore_id={},
    docstore=InMemoryDocstore()
)
vector_store.add_documents(documents)

retriever_vectordb = vector_store.as_retriever(
    search_kwargs={"k": min(10, vector_store.index.ntotal)}
)

In [36]:
from langchain_community.retrievers import BM25Retriever  
keyword_retriever = BM25Retriever.from_documents(documents)
keyword_retriever.k = 10

In [6]:
def compute_accuracy(ground_truth_id, result):
    return 1 if ground_truth_id in result else 0

In [37]:
from langchain.retrievers import EnsembleRetriever

In [38]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever_vectordb, keyword_retriever],
    weights=[0.8, 0.2]
)

In [ ]:
def find_top_k_bm25(queries, query_ids, documents):
    results = {}
    i = 1
    for query, query_id in zip(queries, query_ids):
        # if(i%100 == 0):
        #     print(i)
        result = ensemble_retriever.get_relevant_documents(query)
        results[query_id] = [r.metadata['doc_id'] for r in result]
        i+=1
    return results

In [40]:
def compute_accuracy(ground_truth_id, result):
    return 1 if ground_truth_id in result else 0

def compute_mrr(ground_truth_id, result):
    for rank, id in enumerate(result):
        if id == ground_truth_id:
            return 1 / (rank + 1)
    return 0

In [42]:
results = find_top_k_bm25(queries, query_ids, documents)

100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400


In [43]:
sum_acc = 0
sum_mrr = 0
for query_id in query_ids:
    result = results[query_id]
    for i  in test_dataset['rel_ids'][query_id]:
        ground_truth_ids = i
    sum_acc += compute_accuracy(ground_truth_ids, result)
    sum_mrr += compute_mrr(ground_truth_ids, result)

In [44]:
print(sum_acc/len(queries))
print(sum_mrr/len(queries))

0.9643900657414171
0.8025002173452096


In [ ]:
models = [
    'thang1943/vietnamese-bi-encoder-v2',
    'thang1943/vietnamese-sbert-v2',
    'thang1943/bkcare-embed-v2',
    'thang1943/bge-m3-finetuned'
]

In [ ]:
import json
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore

with open('/mnt/data1tb/thangcn/datnv2/data/alobs_test_dataset.json', encoding='utf8') as f:
    test_dataset = json.load(f)
corpus = []
corpus_ids = []
for id in test_dataset['corpus'].keys():
    corpus.append(test_dataset['corpus'][id])
    corpus_ids.append(id)

corpus_embeddings = embedder.encode(corpus, convert_to_tensor=True)
queries = []
query_ids = []
for id in test_dataset['queries'].keys():
    queries.append(test_dataset['queries'][id])
    query_ids.append(id)

In [ ]:
documents = [Document(page_content=text, metadata={"doc_id": id}) for text, id in zip(corpus,corpus_ids)]

In [ ]:
for model in models:
    embeddings = HuggingFaceEmbeddings(
        model_name=model,
        model_kwargs={'device': 'cuda'}
    )
    index = faiss.IndexFlatL2(len(embeddings.embed_query('thang')))
    vector_store = FAISS(
        embedding_function=embeddings,
        index=index,
        index_to_docstore_id={},
        docstore=InMemoryDocstore()
    )
    vector_store.add_documents(documents)

    retriever_vectordb = vector_store.as_retriever(
        search_kwargs={"k": min(10, vector_store.index.ntotal)}
    )